# Chapter 7 – Context Management
## Hands-On Colab Notebook for Network Engineers

**The story that started this chapter:**

> *"Our branch router automation worked perfectly on 500–2000 line configs.*
> *Then the data center team asked: can we run it on the Nexus 9500?*
> *47,000 lines. ~210,000 tokens. The context window is 200,000.*
> *My first instinct: truncate. Wrong — the ACLs I needed were at the end.*
> *My second instinct: split in half. Also wrong — BGP split in the middle, model said 'incomplete config'.*
> *What I needed was intelligent context management — the same discipline as IP fragmentation."*

| # | Demo | What it teaches |
|---|------|-----------------|
| 1 | Token Counting | Count before you call — never fly blind |
| 2 | Naive Chunking Fails | Why splitting at random bytes breaks analysis |
| 3 | Semantic Chunking | Chunk at natural config boundaries like OSPF area splits |
| 4 | Map-Reduce Pattern | Parallel analysis of large configs → unified report |
| 5 | Prompt Caching | 90% cost reduction for repeated system prompts |
| 6 | Conversation Memory | Rolling summary — keep context without blowing the window |
| 7 | Full Pipeline | Auto-select strategy based on config size |

**~40 minutes** | **~$0.12 in API calls** | **No local setup required**

> Networking analogy: context management = IP fragmentation + reassembly.
> The MTU is the context window. You cannot push a 230,000-token datagram through a 200,000-token link.
> You must fragment intelligently, at natural packet boundaries, with reassembly metadata.


---
## Setup

In [ ]:
!pip install anthropic -q

import json, re, time
from anthropic import Anthropic

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# Context limits — hard walls, not soft suggestions
CONTEXT_LIMITS = {
    HAIKU:  200_000,
    SONNET: 200_000,
}

# ── Sample configs of different sizes ─────────────────────────────────────────
SMALL_CONFIG = """hostname branch-router-01
!
interface GigabitEthernet0/0
 description WAN to ISP
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description LAN
 ip address 10.0.1.1 255.255.255.0
 no shutdown
!
line vty 0 4
 transport input ssh
 login local
!
ntp server 216.239.35.0"""

# Medium config — 30 interfaces, routing, ACLs
# Build without nested triple-quotes to avoid Python syntax issues
_iface_lines = [
    line
    for i in range(1, 31)
    for line in [
        f"interface GigabitEthernet1/0/{i}",
        f" description Server-Port-{i}",
        f" ip address 10.{i//256}.{i%256}.1 255.255.255.0",
        " no shutdown", "!",
    ]
]
MEDIUM_CONFIG = "\n".join([
    "hostname core-switch-01", "!",
    "enable secret 9 $9$abc123",
    "service password-encryption", "!",
] + _iface_lines + [
    "!", "router ospf 1",
    " network 10.0.0.0 0.255.255.255 area 0",
    " passive-interface default",
    " no passive-interface GigabitEthernet1/0/1",
    "!", "ip access-list extended BLOCK_TELNET",
    " deny tcp any any eq 23", " permit ip any any",
    "!", "snmp-server community public RO",
    "line vty 0 15", " transport input ssh",
    " exec-timeout 5 0", " login local",
])

print("Setup complete.")
print(f"Small config:  {len(SMALL_CONFIG.split(chr(10))):>5} lines  (~{len(SMALL_CONFIG)//4:,} tokens est.)")
print(f"Medium config: {len(MEDIUM_CONFIG.split(chr(10))):>5} lines  (~{len(MEDIUM_CONFIG)//4:,} tokens est.)")


---
## Demo 1 — Token Counting: Never Fly Blind

**Rule 1 of context management: count tokens before you call the API.**

A failed API call costs the same as a successful one.
A call that silently truncates your config costs you a false sense of security.

The Anthropic `count_tokens` API is:
- **Free** — much cheaper than an actual call
- **Accurate** — uses the same tokenizer as the real call
- **Fast** — returns in milliseconds

> Networking analogy: this is the packet-size check before transmission.
> You don't send first and hope. You measure, then decide.


In [ ]:
def count_tokens_for_config(config: str, system: str = "", model: str = SONNET) -> dict:
    """
    Count tokens accurately using Claude's official count_tokens API.
    Returns a breakdown: content tokens, system tokens, total, and whether it fits.
    """
    limit = CONTEXT_LIMITS.get(model, 200_000)
    output_reserve = 2000      # tokens reserved for the response
    safety_buffer  = 10_000    # 5% buffer — don't push to the absolute edge

    messages = [{"role": "user", "content": config}]

    if system:
        resp = client.messages.count_tokens(model=model, system=system, messages=messages)
        # Estimate system tokens separately (quick approximation)
        sys_resp = client.messages.count_tokens(
            model=model, messages=[{"role": "user", "content": system}]
        )
        system_tokens  = sys_resp.input_tokens
        content_tokens = resp.input_tokens - system_tokens
        total_needed   = resp.input_tokens + output_reserve
    else:
        resp = client.messages.count_tokens(model=model, messages=messages)
        system_tokens  = 0
        content_tokens = resp.input_tokens
        total_needed   = resp.input_tokens + output_reserve

    effective_limit = limit - safety_buffer
    fits = total_needed <= effective_limit
    utilization = (total_needed / limit) * 100

    return {
        "fits":            fits,
        "content_tokens":  content_tokens,
        "system_tokens":   system_tokens,
        "output_reserve":  output_reserve,
        "total_needed":    total_needed,
        "limit":           limit,
        "effective_limit": effective_limit,
        "utilization_pct": utilization,
        "overflow":        max(0, total_needed - effective_limit),
    }

def recommend_strategy(result: dict) -> str:
    """Tell the user what to do based on the token count result."""
    if result["fits"] and result["utilization_pct"] < 70:
        return "✅ Direct API call — plenty of headroom"
    elif result["fits"] and result["utilization_pct"] < 90:
        return "⚠️  Direct call but close — consider prompt caching"
    elif result["fits"]:
        return "⚠️  Fits but >90% used — enable prompt caching, monitor carefully"
    elif result["overflow"] < 20_000:
        return f"❌ Over by {result['overflow']:,} tokens — try smaller output_reserve or trim content"
    else:
        return f"❌ Over by {result['overflow']:,} tokens — use Map-Reduce chunking"

SYSTEM_PROMPT = "You are a senior Cisco network security engineer. Analyze configs thoroughly."

print("=== Token Count Check ===\n")
for label, config in [("Small config (branch router)", SMALL_CONFIG),
                       ("Medium config (core switch)",  MEDIUM_CONFIG)]:
    result = count_tokens_for_config(config, system=SYSTEM_PROMPT)
    print(f"{label}:")
    print(f"  Content:    {result['content_tokens']:>8,} tokens")
    print(f"  System:     {result['system_tokens']:>8,} tokens")
    print(f"  Output res: {result['output_reserve']:>8,} tokens")
    print(f"  Total:      {result['total_needed']:>8,} / {result['effective_limit']:,} ({result['utilization_pct']:.1f}%)")
    print(f"  Decision:   {recommend_strategy(result)}")
    print()


---
## Demo 2 — Why Naive Chunking Breaks Everything

**Splitting at arbitrary character positions is like cutting a TCP segment mid-header.**

The receiver (the model) cannot reassemble the packet because the header is split.

Watch what happens when we naively split a config at the halfway point.
The model analyzing each half gets an incomplete picture — and gives wrong answers.


In [ ]:
# This is the WRONG way — do not do this in production
def naive_chunk(text: str, pieces: int = 2) -> list:
    """Split at fixed character position — no regard for logical boundaries."""
    size = len(text) // pieces
    return [text[i:i+size] for i in range(0, len(text), size)]

# Build a config that has an ACL at the end (realistic — Cisco puts ACLs last)
TEST_CONFIG = """hostname test-router
!
interface GigabitEthernet0/0
 ip address 10.0.0.1 255.255.255.0
 no shutdown
!
interface GigabitEthernet0/1
 ip address 10.0.1.1 255.255.255.0
 no shutdown
!
router ospf 1
 network 10.0.0.0 0.255.255.255 area 0
!
line vty 0 4
 transport input ssh
 exec-timeout 5 0
 login local
!
ip access-list extended MGMT_ACCESS
 permit tcp 10.100.0.0 0.0.0.255 any eq 22
 deny   tcp any any eq 23
 deny   tcp any any eq 21
 deny   udp any any eq 161
 permit ip any any
log"""

chunks = naive_chunk(TEST_CONFIG, pieces=2)

print("=== Naive 50/50 split result ===\n")
for i, chunk in enumerate(chunks):
    print(f"CHUNK {i+1} (last 120 chars shown):")
    print(f"  '...{chunk[-120:].strip()}'")
    print()

# Ask the model to analyze each chunk
def quick_ask(text: str, question: str) -> str:
    resp = client.messages.create(
        model=HAIKU, max_tokens=120, temperature=0,
        messages=[{"role": "user",
                   "content": f"Config fragment:\n{text}\n\n{question}"}]
    )
    return resp.content[0].text.strip()

print("=== Model analysis of each chunk ===\n")
q = "Is there a complete ACL configuration in this fragment? Reply with YES/NO and one sentence why."
for i, chunk in enumerate(chunks):
    answer = quick_ask(chunk, q)
    print(f"Chunk {i+1}: {answer}\n")

print("The correct answer is: YES, there is a complete ACL — but naive splitting hid it.")
print("Chunk 1 ends mid-config. Chunk 2 starts mid-ACL. Neither is analyzable correctly.")


### Why This Matters

Naive chunking is the #1 mistake engineers make with large configs.
When you split at arbitrary byte positions:

- An ACL might get split between `permit` and `deny` rules
- A BGP config might lose its `address-family` context
- A VRF definition might be separated from its route-targets

> **The fix:** Always split at **logical section boundaries** (the `!` separators
> in Cisco configs). This is exactly what Demo 3 does — semantic chunking.

---
## Demo 3 — Semantic Chunking: Respect the Config Structure

**Chunk at natural boundaries — like OSPF area splits.**

Cisco IOS/IOS-XE configs follow a clear hierarchical structure:
- `interface ...` blocks → complete, self-contained
- `router ospf/bgp ...` blocks → complete routing protocol config
- `ip access-list ...` blocks → complete ACL
- `line vty ...` → complete VTY config

Each section is logically independent.
Splitting between sections (at `!` separators) preserves full semantic meaning.

> Like how OSPF floods complete LSAs within an area — you never send half an LSA.


In [ ]:
def semantic_chunk(config: str) -> list:
    """
    Split a Cisco IOS config at logical section boundaries.
    Each chunk is a complete, coherent block — never split mid-section.

    Returns list of dicts: {section_type, name, content, est_tokens}
    """
    # What starts a new section in a Cisco config
    SECTION_MARKERS = [
        (r'^interface\s+',         'interface'),
        (r'^router bgp\s+',        'bgp'),
        (r'^router ospf\s+',       'ospf'),
        (r'^router\s+',            'routing'),
        (r'^ip access-list\s+',    'acl'),
        (r'^access-list\s+',       'acl'),
        (r'^ip prefix-list\s+',    'prefix_list'),
        (r'^route-map\s+',         'route_map'),
        (r'^line\s+(vty|con|aux)', 'line'),
        (r'^snmp-server',           'snmp'),
        (r'^ntp\s+',               'ntp'),
        (r'^crypto\s+',            'crypto'),
        (r'^aaa\s+',               'aaa'),
    ]

    chunks = []
    lines  = config.split('\n')
    current_type  = 'global'
    current_name  = 'global_config'
    current_lines = []

    def flush():
        if current_lines:
            content = '\n'.join(current_lines).strip()
            if content:
                chunks.append({
                    "section_type": current_type,
                    "name":         current_name,
                    "content":      content,
                    "est_tokens":   len(content) // 4,
                })

    for line in lines:
        stripped = line.strip()
        new_type = None
        for pattern, stype in SECTION_MARKERS:
            if re.match(pattern, stripped, re.IGNORECASE):
                new_type = stype
                break

        if new_type is not None:
            flush()
            current_type  = new_type
            current_name  = stripped[:60]      # first 60 chars as section name
            current_lines = [line]
        else:
            current_lines.append(line)

    flush()   # Don't forget the last section
    return chunks

chunks = semantic_chunk(TEST_CONFIG)

print(f"=== Semantic chunking result: {len(chunks)} sections ===\n")
for ch in chunks:
    print(f"  [{ch['section_type']:<12}]  {ch['name'][:50]:<50}  ~{ch['est_tokens']} tokens")

print()
print("=== Model analysis of the ACL chunk (complete section) ===\n")
acl_chunks = [c for c in chunks if c['section_type'] == 'acl']
if acl_chunks:
    acl = acl_chunks[0]
    q2  = "Describe what this ACL does. Is it permitting or denying Telnet (port 23)?"
    answer = quick_ask(acl['content'], q2)
    print(f"ACL chunk analysis:\n{answer}")
else:
    print("No ACL found — adjust the section markers.")


---
## Demo 4 — Map-Reduce: Parallel Analysis of Large Configs

**When the config is too large, analyze chunks in parallel — then synthesize.**

Map-Reduce for configs:
1. **SPLIT**: chunk config at semantic boundaries
2. **MAP**: analyze each chunk independently (can run in parallel)
3. **REDUCE**: combine all chunk findings into a unified report

This is the same pattern OSPF uses:
- Each area runs SPF independently (MAP)
- ABRs summarize and aggregate routes (REDUCE)
- Core sees only summaries, not every individual LSA

Why parallel? 10 chunks × 2 seconds each = 2 seconds total, not 20.


In [ ]:
import concurrent.futures

def analyze_chunk(chunk: dict, idx: int, total: int) -> dict:
    """
    MAP step: Analyze a single config section for security issues.
    Returns a structured dict of findings for just this section.
    """
    prompt = f"""Analyze this Cisco config section for security issues.
Section type: {chunk['section_type']}
Section {idx+1} of {total}

Config:
{chunk['content']}

Return JSON only (no other text):
{{
  "section": "{chunk['section_type']}",
  "idx": {idx},
  "critical": ["list critical issues"],
  "warnings": ["list warnings"],
  "ok":        ["list correctly configured items"]
}}"""

    for attempt in range(2):
        try:
            resp = client.messages.create(
                model=HAIKU, max_tokens=400, temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            raw = resp.content[0].text
            start = raw.find('{'); end = raw.rfind('}') + 1
            if start >= 0:
                return json.loads(raw[start:end])
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                return {"section": chunk['section_type'], "idx": idx,
                        "critical": [], "warnings": [], "ok": [], "error": str(e)}
    return {"section": chunk['section_type'], "idx": idx,
            "critical": [], "warnings": [], "ok": []}


def reduce_findings(results: list, device: str) -> dict:
    """
    REDUCE step: Combine all chunk findings into a unified report.
    Uses Haiku for the simple aggregation + one Sonnet call for the summary.
    """
    all_critical = []
    all_warnings = []
    all_ok       = []

    for r in results:
        all_critical.extend(r.get("critical", []))
        all_warnings.extend(r.get("warnings", []))
        all_ok.extend(r.get("ok", []))

    # Deduplicate (overlap chunks can report the same issue twice)
    all_critical = list(dict.fromkeys(all_critical))
    all_warnings = list(dict.fromkeys(all_warnings))

    # One Sonnet call to write the executive summary
    summary_prompt = f"""Device: {device}
Security findings:
  Critical ({len(all_critical)}): {all_critical}
  Warnings ({len(all_warnings)}): {all_warnings}

Write a 2-sentence executive summary of the security posture.
Be direct. State the overall risk level first."""

    summary_resp = client.messages.create(
        model=SONNET, max_tokens=150, temperature=0,
        messages=[{"role": "user", "content": summary_prompt}]
    )

    risk = min(100, len(all_critical) * 30 + len(all_warnings) * 10)
    return {
        "device":    device,
        "risk_score": risk,
        "critical":  all_critical,
        "warnings":  all_warnings,
        "ok":        all_ok[:5],    # Top 5 passing checks
        "summary":   summary_resp.content[0].text.strip(),
        "chunks_analyzed": len(results),
    }


def map_reduce_analysis(config: str, device: str = "unknown") -> dict:
    """Full Map-Reduce pipeline: split → parallel analyze → reduce."""
    chunks = semantic_chunk(config)
    print(f"Config split into {len(chunks)} sections. Analyzing in parallel...")

    # MAP — parallel using ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        futures = {
            pool.submit(analyze_chunk, c, i, len(chunks)): i
            for i, c in enumerate(chunks)
        }
        results = []
        for f in concurrent.futures.as_completed(futures):
            try:
                results.append(f.result())
            except Exception as e:
                print(f"  Chunk failed: {e}")

    results.sort(key=lambda x: x.get("idx", 0))  # re-order by index

    # REDUCE
    return reduce_findings(results, device)


print("Running Map-Reduce analysis on MEDIUM_CONFIG...")
t0 = time.time()
report = map_reduce_analysis(MEDIUM_CONFIG, device="core-switch-01")
elapsed = time.time() - t0

print(f"\n=== Map-Reduce Report ===")
print(f"Device:          {report['device']}")
print(f"Risk Score:      {report['risk_score']}/100")
print(f"Chunks analyzed: {report['chunks_analyzed']}")
print(f"Critical issues: {len(report['critical'])}")
for c in report['critical']:
    print(f"  ❌ {c}")
print(f"Warnings:        {len(report['warnings'])}")
for w in report['warnings'][:3]:
    print(f"  ⚠️  {w}")
print(f"\nSummary: {report['summary']}")
print(f"\nTotal time: {elapsed:.1f}s")


### Map-Reduce in Plain Language

```
Big config (too large for one API call)
      │
      ▼
Split into sections (semantic_chunk)
      │
  ┌───┼───┐
  ▼   ▼   ▼
 Analyze each section independently (MAP — can run in parallel)
  │   │   │
  └───┼───┘
      ▼
Combine all findings into one report (REDUCE)
```

This is the same pattern OSPF uses: each area runs SPF independently,
then ABRs summarize routes across areas. The core never sees individual LSAs
from other areas — just the summary.

---
## Demo 5 — Prompt Caching: 90% Cost Reduction

**If your system prompt doesn't change between calls, you're paying for it every time.**

Prompt caching works like a network cache:
- First call: processes and stores the system prompt in the KV cache (cache WRITE)
- Subsequent calls: reuses the stored cache instead of reprocessing (cache HIT)
- Result: 90% cheaper, 75% faster for the cached tokens

**What to cache**: system prompts, long instructions, reference documents, few-shot examples.
**What NOT to cache**: the actual config being analyzed (different every call).

Anthropic pricing:
- Cache write: 1.25× normal (one-time cost)
- Cache read:  0.10× normal (90% discount every subsequent call)


In [ ]:
# The large system prompt we want to cache — same for every device we analyze
SECURITY_SYSTEM = """You are a senior Cisco IOS-XE security compliance engineer.

METHODOLOGY — check in this order:
1. Authentication: enable secret (type 9), no enable password
2. Remote access: VTY lines must use transport input ssh ONLY, exec-timeout set
3. SNMP: v1/v2c = CRITICAL risk, v3 with auth+priv = compliant
4. Password encryption: service password-encryption must be configured
5. Logging: syslog server configured, logging buffered for local retention
6. Unused services: no ip http server (unless needed), no service finger
7. NTP: ntp server configured with authentication preferred

SEVERITY:
CRITICAL = remotely exploitable / allows unauthorized access
HIGH     = increases attack surface, requires local access to exploit
MEDIUM   = best practice deviation

RESPONSE FORMAT: JSON only.
{
  "critical": [{"issue": "...", "evidence": "...", "fix": "..."}],
  "high":     [{"issue": "...", "evidence": "...", "fix": "..."}],
  "medium":   [{"issue": "...", "fix": "..."}],
  "ok":       ["passing checks"],
  "risk":     0-100
}"""

def analyze_with_caching(hostname: str, config: str) -> dict:
    """
    Analyze a device config using a cached system prompt.
    The system prompt is marked cache_control='ephemeral'.
    First call writes the cache. All subsequent calls read from it (90% cheaper).
    """
    response = client.messages.create(
        model=SONNET,
        max_tokens=800,
        temperature=0,
        system=[
            {
                "type": "text",
                "text": SECURITY_SYSTEM,
                "cache_control": {"type": "ephemeral"}   # Mark for caching
            }
        ],
        messages=[{
            "role": "user",
            "content": f"Security audit for device '{hostname}':\n\n{config}"
        }]
    )

    # Read cache statistics from the usage object
    cache_read  = getattr(response.usage, 'cache_read_input_tokens', 0)
    cache_write = getattr(response.usage, 'cache_creation_input_tokens', 0)
    input_tok   = response.usage.input_tokens
    output_tok  = response.usage.output_tokens

    if cache_read > 0:
        status = f"HIT  ({cache_read:,} tokens from cache)"
    elif cache_write > 0:
        status = f"WRITE ({cache_write:,} tokens cached for next calls)"
    else:
        status = "MISS  (no cache activity)"

    # Parse the JSON response
    raw   = response.content[0].text
    start = raw.find('{'); end = raw.rfind('}') + 1
    result = json.loads(raw[start:end]) if start >= 0 else {}
    result.update({
        "device":       hostname,
        "cache_status": status,
        "input_tokens": input_tok,
        "output_tokens": output_tok,
    })
    return result

# Simulate analyzing multiple branch devices (same system prompt, different configs)
branch_configs = {
    "branch-01": SMALL_CONFIG,
    "branch-02": SMALL_CONFIG.replace("branch-router-01", "branch-router-02"),
    "branch-03": SMALL_CONFIG.replace("branch-router-01", "branch-router-03"),
}

print("=== Prompt Caching Demo ===")
print(f"System prompt size: ~{len(SECURITY_SYSTEM)//4} tokens")
print()
print(f"{'Device':<12}  {'Cache':<40}  {'Risk':>4}  {'Tokens':>7}")
print("-" * 70)

for hostname, config in branch_configs.items():
    r = analyze_with_caching(hostname, config)
    risk = r.get('risk', '?')
    tok  = r.get('input_tokens', 0) + r.get('output_tokens', 0)
    print(f"{hostname:<12}  {r['cache_status']:<40}  {str(risk):>4}  {tok:>7,}")

print()
print("Call 1: system prompt written to cache (WRITE)")
print("Call 2: system prompt read from cache (HIT) — 90% cheaper for those tokens")
print("Call 3: same — cache still warm")
print()
print("In production with 100 devices/day and a 1,000-token system prompt:")
sys_tokens = len(SECURITY_SYSTEM) // 4
print(f"  Without caching: 100 × {sys_tokens} = {100*sys_tokens:,} tokens/day at full price")
print(f"  With caching:    1 write + 99 reads = ~{sys_tokens + 99*(sys_tokens//10):,} effective tokens/day")
print(f"  Savings: ~{100 - (sys_tokens + 99*(sys_tokens//10)) * 100 // (100*sys_tokens)}%")


---
## Demo 6 — Conversation Memory: Rolling Summary

**Multi-turn sessions grow without bound. Manage them or hit the wall.**

The problem: each turn adds 100–500 tokens to the conversation history.
Turn 30 = 15,000+ tokens of history before the actual question.

The solution: rolling summary.
- Keep the last N turns verbatim (immediately relevant)
- Compress older turns into a 3-sentence summary
- The model sees the summary + recent context

> Like how OSPF uses LSA aging — old topology info gets flushed.
> You remember the key facts (key decisions, findings), not every word.


In [ ]:
class NetworkChatSession:
    """
    Multi-turn chat with automatic context compression.
    Uses rolling summarization to stay within token budget.
    """

    def __init__(self, system: str, model: str = SONNET,
                 max_context_tokens: int = 80_000,
                 recent_turns_to_keep: int = 6):
        self.system                = system
        self.model                 = model
        self.max_context_tokens    = max_context_tokens
        self.recent_turns_to_keep  = recent_turns_to_keep
        self.messages              = []
        self.episode_summary       = ""   # compressed history
        self.total_turns           = 0

    def _est_tokens(self, text: str) -> int:
        return len(text) // 4   # fast rough estimate

    def _context_size(self) -> int:
        total  = self._est_tokens(self.system)
        total += self._est_tokens(self.episode_summary)
        for m in self.messages:
            total += self._est_tokens(str(m.get("content", "")))
        return total

    def _compress_history(self):
        """
        Compress older turns into a concise summary.
        Keep the last N turns verbatim — they're immediately relevant.
        """
        keep = self.recent_turns_to_keep * 2   # user+assistant pairs
        if len(self.messages) <= keep:
            return

        old_turns  = self.messages[:-keep]
        new_recent = self.messages[-keep:]

        old_text = "\n".join([
            f"{m['role'].upper()}: {m['content']}"
            for m in old_turns if isinstance(m.get('content'), str)
        ])

        compress_prompt = f"""Summarize this network troubleshooting conversation.
Preserve: device names, IPs, commands run, findings, conclusions.
Discard: filler, repetition, greetings.
Max 4 sentences.

Previous summary (if any): {self.episode_summary or 'None'}

Conversation:
{old_text[:3000]}

Summary:"""

        resp = client.messages.create(
            model=HAIKU, max_tokens=200, temperature=0,
            messages=[{"role": "user", "content": compress_prompt}]
        )
        self.episode_summary = resp.content[0].text.strip()
        self.messages        = list(new_recent)

    def chat(self, user_msg: str) -> str:
        """Send a message, manage context, return the response."""
        self.total_turns += 1
        self.messages.append({"role": "user", "content": user_msg})

        # Check if we need to compress before calling
        if self._context_size() > self.max_context_tokens * 0.8:
            before = self._context_size()
            self._compress_history()
            after  = self._context_size()
            print(f"  [Context compressed: {before:,} → {after:,} tokens]")

        # Build system: episode summary prepended so model remembers older context
        system_content = self.system
        if self.episode_summary:
            system_content = (
                f"CONVERSATION HISTORY SUMMARY:\n{self.episode_summary}\n\n---\n"
                + self.system
            )

        resp = client.messages.create(
            model=self.model, max_tokens=500, temperature=0,
            system=system_content, messages=self.messages
        )
        answer = resp.content[0].text
        self.messages.append({"role": "assistant", "content": answer})
        return answer

    def stats(self) -> str:
        return (f"Turn {self.total_turns} | "
                f"{len(self.messages)} msgs in context | "
                f"~{self._context_size():,} tokens | "
                f"{'has summary' if self.episode_summary else 'no summary yet'}")

# Simulate a realistic troubleshooting session
session = NetworkChatSession(
    system="You are a senior network operations engineer. Help troubleshoot network issues concisely.",
    recent_turns_to_keep=3
)

conversation = [
    "We're seeing high CPU on our core router core-01 (10.0.0.1). Where do we start?",
    "The CPU is at 89%. BGP process is consuming 45% of it. The router has 12 BGP peers.",
    "show bgp summary shows 3 peers in Active state: 10.1.1.1, 10.1.1.5, 10.1.1.9",
    "After checking, 10.1.1.1 has an ASN mismatch. Our side says AS65001, remote says AS65002.",
    "Fixed the ASN config. BGP came up for 10.1.1.1. CPU dropped to 52%. Still high.",
    "The remaining CPU load is from the OSPF process now. No changes made to OSPF recently.",
]

print("=== Multi-Turn Troubleshooting Session ===\n")
for i, user_msg in enumerate(conversation):
    print(f"Engineer (turn {i+1}): {user_msg[:70]}...")
    answer = session.chat(user_msg)
    print(f"AI: {answer[:150].strip()}...")
    print(f"   [{session.stats()}]\n")

print("=== Session Complete ===")
print(f"Total turns: {session.total_turns}")
print(f"Messages in active context: {len(session.messages)} (vs {session.total_turns*2} without compression)")
if session.episode_summary:
    print(f"\nEpisode summary (compressed memory):")
    print(f"  {session.episode_summary}")


---
## Demo 7 — Full Pipeline: Auto-Select Strategy

**Production-ready: count tokens first, then pick the right strategy automatically.**

```
Config received
       │
       ▼
Count tokens (free, fast)
       │
       ├─── < 70% used ──→ Direct API call
       │
       ├─── 70–90% used ─→ Direct call + prompt caching
       │
       └─── > 100% used ─→ Semantic chunk + Map-Reduce
```

One function handles all three cases. Zero guessing. Zero manual calculation.


In [ ]:
def process_config_smart(device: str, config: str,
                          system: str = SECURITY_SYSTEM) -> dict:
    """
    Auto-select the right strategy based on config size.
    Transparent: prints which strategy it chose and why.
    """
    print(f"\nProcessing: {device}")
    print(f"  Config size: {len(config.split(chr(10))):,} lines, ~{len(config)//4:,} tokens est.")

    # Step 1: count tokens (free)
    check = count_tokens_for_config(config, system=system)
    print(f"  Token check: {check['total_needed']:,} / {check['effective_limit']:,} "
          f"({check['utilization_pct']:.0f}%) — fits={check['fits']}")

    if not check['fits']:
        # Strategy C: Map-Reduce
        print(f"  Strategy: Map-Reduce (overflow: {check['overflow']:,} tokens)")
        return map_reduce_analysis(config, device)

    elif check['utilization_pct'] > 70:
        # Strategy B: direct call with caching
        print("  Strategy: Direct call + prompt caching")
        return analyze_with_caching(device, config)

    else:
        # Strategy A: simple direct call
        print("  Strategy: Direct API call (small config)")
        resp = client.messages.create(
            model=HAIKU, max_tokens=500, temperature=0,
            system=system,
            messages=[{"role": "user",
                       "content": f"Security audit for {device}:\n\n{config}"}]
        )
        raw   = resp.content[0].text
        start = raw.find('{'); end = raw.rfind('}') + 1
        result = json.loads(raw[start:end]) if start >= 0 else {"note": raw}
        result["device"] = device
        return result


# Run on two different sizes
print("=" * 60)
for device, config in [("branch-router-01", SMALL_CONFIG),
                        ("core-switch-01",   MEDIUM_CONFIG)]:
    report = process_config_smart(device, config)
    risk   = report.get('risk', report.get('risk_score', '?'))
    crit   = len(report.get('critical', []))
    print(f"  Result: risk={risk}/100, critical={crit}")


---
## Summary

**The five strategies — and when to use each:**

| Situation | Strategy | Code pattern |
|-----------|----------|-------------|
| Config < 70% of context | Direct call | Simple `messages.create()` |
| Config 70–90% of context | Direct + caching | Add `cache_control` to system prompt |
| Config > 100% of context | Map-Reduce | `semantic_chunk()` → parallel analyze → `reduce_findings()` |
| Multi-turn session | Rolling summary | `NetworkChatSession._compress_history()` |
| Always | Count first | `count_tokens()` before every large call |

**The MTU analogy in one sentence:**
Your context window is the MTU. Token counting is PMTUD.
Semantic chunking is fragmentation at packet boundaries.
Map-Reduce is reassembly with deduplication.
Rolling summary is the sliding window that keeps context alive without flooding the channel.

> Always count tokens before you call.
> Always chunk at semantic boundaries, not arbitrary character positions.
> Always cache system prompts that don't change between calls.
